# Go2 ODD/COD Observer - Complete Workflow

This notebook demonstrates the complete workflow for analyzing Operational Design Domain (ODD) compliance and Conditions of Deployment (COD) for Unitree Go2 robot scenarios.

**Workflow Overview:**
1. Setup dependencies and configure Google AI SDK
2. Define ODD specifications in natural language
3. Instantiate multi-modal AI agents (Motion, Image, LiDAR, Collision)
4. Load and process scenario data
5. Evaluate ODD compliance and compute distance metrics
6. Visualize results and generate reports

**Note:** This workflow assumes you have preprocessed ROS2 bag files into time-windowed snapshots using the `extract_windows.py` script.

## 1. Setup and Dependencies

Install and import required packages for Google AI SDK and our analysis framework.

In [1]:
# Install Google Agent Development Kit (ADK) and dependencies
# Note: Run this cell only once or when packages need updating
!pip install -q google-adk python-dotenv

In [2]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Standard library imports
import json
import base64
import io
from typing import Dict, List, Tuple, Any

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Google ADK imports
from google.genai import types
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, ToolContext

/usr/local/python/3.10.19/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## 2. Configuration (REQUIRED)

### 2.1 Google Gemini API Key

**This notebook requires a Google Gemini API key** to demonstrate AI agents in action.

Get your free API key at: https://aistudio.google.com/app/apikey

### 2.2 Model Selection

Choose which Gemini model to use for all agents:
- `gemini-2.0-flash-lite`: **Recommended** - 30 RPM free tier, fastest
- `gemini-2.0-flash`: 15 RPM free tier, balanced
- `gemini-2.5-flash`: Latest flash - 10 RPM free tier
- `gemini-2.5-pro`: Most capable - 2 RPM free tier (slower)

In [3]:
# ============================================
# 2.1 Configure Google API Key
# ============================================
import os
from dotenv import load_dotenv

# Option 1: Set via environment variable (RECOMMENDED)
# export GOOGLE_API_KEY='your-api-key-here'

# Option 2: Load from .env file
load_dotenv()

# Option 3: Set directly in notebook (NOT recommended - avoid committing keys!)
# os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'

# Verify API key is configured
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    print("✓ Google AI SDK configured successfully")
    print("  API key detected")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("This notebook REQUIRES a Google Gemini API key to run.")
    print()
    print("To get a free API key:")
    print("  1. Visit https://aistudio.google.com/app/apikey")
    print("  2. Create or select a project")
    print("  3. Generate an API key")
    print()
    print("  To configure your API key:")
    print("  export GOOGLE_API_KEY='your-key-here'")
    print("  OR create a .env file with: GOOGLE_API_KEY=your-key-here")
    print()
    print("⚠ The notebook will FAIL without an API key - this is intentional!")
    print("  Falling back to fake data would defeat the purpose of learning about AI agents.")

# ============================================
# 2.2 Model Configuration
# ============================================
# Change this to switch all agents to a different model
GEMINI_MODEL = "gemini-2.0-flash-lite"  # Recommended for free tier (30 RPM)
# GEMINI_MODEL = "gemini-2.0-flash"      # Balanced (15 RPM)
# GEMINI_MODEL = "gemini-2.5-flash"      # Latest (10 RPM)
# GEMINI_MODEL = "gemini-2.5-pro"        # Most capable (2 RPM - slower)

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

print(f"✓ Using model: {GEMINI_MODEL}")
print("✓ Agent config: JSON response format, temperature=0.1")

✓ Google AI SDK configured successfully
  API key detected
✓ Using model: gemini-2.0-flash-lite
✓ Agent config: JSON response format, temperature=0.1


## 3. User Inputs

Define what you want to analyze:
1. **Natural language ODD**: Operating constraints in plain English
2. **Dataset path**: Location of preprocessed window data

The orchestrator agent (Section 5) will handle everything from here.

## 3.5 Pre-load Scenario Data (Local Kernel)

Since cloud agents can't access local files, we pre-load all data locally and pass to agents via session state.
Uses raw image bytes (not base64) for native Gemini vision support.


In [4]:
# ============================================================================
# PRE-LOAD SCENARIO DATA LOCALLY
# ============================================================================
# Load CSV + images from disk, prepare for agent consumption
# This happens in notebook kernel (has file access), not in cloud agents

def load_window_data(scenario_path: Path, window_id: str, run_id: str = None) -> Tuple[Dict, bytes, Dict[str, bytes]]:
    """Load motion JSON, camera PNG bytes, and BEV PNG bytes for a window."""
    if run_id is None:
        run_id = scenario_path.name
    
    motion_file = scenario_path / f"motion_{run_id}_w{window_id}.json"
    camera_file = scenario_path / f"cam_{run_id}_w{window_id}.png"
    
    # Load motion JSON
    with open(motion_file, 'r') as f:
        motion_data = json.load(f)
    
    # Load camera as raw PNG bytes
    with open(camera_file, 'rb') as f:
        camera_bytes = f.read()
    
    # Load BEV images as raw PNG bytes
    bev_bytes = {}
    for channel in ['occupancy', 'height', 'density', 'roughness']:
        bev_file = scenario_path / f"bev_{channel}_{run_id}_w{window_id}.png"
        if bev_file.exists():
            with open(bev_file, 'rb') as f:
                bev_bytes[channel] = f.read()
    
    return motion_data, camera_bytes, bev_bytes

def load_scenario_index(scenario_path: Path) -> pd.DataFrame:
    """Load window index CSV."""
    index_files = list(scenario_path.glob("index_*.csv"))
    if not index_files:
        raise FileNotFoundError(f"No index file found in {scenario_path}")
    return pd.read_csv(index_files[0])

# Initialize image data map (will be populated when preload is called)
image_data_map = {}
scenario_data = {}  # Will be populated below

def preload_scenario_data(scenario_path: Path):
    """Load all windows from scenario directory into memory."""
    global scenario_data, image_data_map
    
    print("=" * 80)
    print("PRE-LOADING SCENARIO DATA (LOCAL KERNEL)")
    print("=" * 80)
    
    index_df = load_scenario_index(scenario_path)
    print(f"\n✓ Loaded index: {len(index_df)} windows")
    
    scenario_data = {
        "scenario": scenario_path.name,
        "windows": []
    }
    
    for _, row in index_df.iterrows():
        window_id = str(row['window_id']).zfill(3)
        try:
            motion_json, camera_bytes, bev_bytes = load_window_data(scenario_path, window_id, scenario_path.name)
            
            # Store motion as JSON (semantic data for agents)
            scenario_data["windows"].append({
                "window_id": window_id,
                "motion_json": motion_json
            })
            
            # Store images in local map for tool access (raw bytes, not serialized)
            image_data_map[window_id] = {
                "camera": camera_bytes,
                **{f"bev_{channel}": img_bytes for channel, img_bytes in bev_bytes.items()}
            }
            print(f"  ✓ Window {window_id}: motion + camera + 4 BEV channels")
        except Exception as e:
            print(f"  ✗ Window {window_id}: {e}")
    
    if scenario_data["windows"]:
        # Get size info from first window
        sample_window_id = scenario_data["windows"][0]["window_id"]
        camera_size = len(image_data_map[sample_window_id]["camera"]) / 1024
        bev_size = sum(len(b) for b in image_data_map[sample_window_id].items() if b[0].startswith("bev_")) / 1024
        print(f"\n✅ Total windows pre-loaded: {len(scenario_data['windows'])}")
        print(f"   Camera size: ~{camera_size:.0f} KB per window")
        print(f"   BEV size: ~{bev_size:.0f} KB per window")
    else:
        print("\n❌ No windows loaded!")

print("✓ Preload function defined (will be called after scenario_path is set)")


✓ Preload function defined (will be called after scenario_path is set)


In [5]:
# Natural language ODD definition for Unitree Go2 indoor navigation

odd_natural_language = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Speed Limits:
   - The robot shall operate at forward velocities between 0 and 1.5 m/s under normal conditions
   - Speeds up to 1.8 m/s are acceptable near the boundary but should trigger warnings
   - The absolute physical limit is 2.5 m/s and must never be exceeded

2. Orientation Limits:
   - Roll and pitch angles must remain within ±15 degrees during normal operation
   - Angles up to ±20 degrees are acceptable at the boundary
   - The robot must never exceed ±30 degrees of roll or pitch

3. Terrain Requirements:
   - The robot is designed for smooth and moderate terrain (office floors, carpet)
   - Rough terrain is outside the operational design domain
   - Very rough terrain is completely prohibited

4. Lighting Conditions:
   - The robot can operate in bright and dim lighting conditions
   - Dark environments are outside the ODD and require additional equipment

5. Human Safety:
   - Humans may be visible at a distance (no restriction)
   - Humans in very close proximity (< 1 meter) violate the ODD
   - The system must maintain safe distances from people

6. Collision Policy:
   - Zero collisions are tolerated - any collision is an ODD violation
   - The system must detect and avoid all obstacles

IMPORTANCE WEIGHTS (for distance computation):
- Collision avoidance: Highest priority (weight: 2.0)
- Human proximity: Very high priority (weight: 1.5)
- Roll/Pitch stability: High priority (weight: 1.2)
- Speed limits: Standard priority (weight: 1.0)
- Terrain type: Standard priority (weight: 1.0)
- Lighting conditions: Lower priority (weight: 0.8)
"""

# Dataset selection
DATA_DIR = Path("data/processed/runs")
scenario_path = DATA_DIR / "sim_run_test"  # Change this to analyze different datasets

print("✓ User inputs configured")
print(f"  - ODD: {len(odd_natural_language)} characters")
print(f"  - Dataset: {scenario_path}")

# PRE-LOAD DATA NOW THAT scenario_path IS DEFINED
preload_scenario_data(scenario_path)

✓ User inputs configured
  - ODD: 1699 characters
  - Dataset: data/processed/runs/sim_run_test


## 4. Define Tool Functions for Agents

These Python functions will be available as tools for the orchestrator agent to call.
They provide access to: file I/O, ODD spec construction, COD computation, and visualization.

In [ ]:
# ============================================================================
# DATA LOADING (already done in local kernel above)
# ============================================================================
# No tools needed for I/O - all files loaded locally into scenario_data dict
# This keeps the architecture clean: tools are for computation, I/O is local


# ============================================================================
# VISUALIZATION TOOLS (for Report Agent)
# ============================================================================

def generate_distance_plot(times: List[float], distances: List[float], title: str = "ODD Distance over Time") -> dict:
    """
    Generate a timeline plot showing how close the robot is to violating the ODD.
    
    Use this tool to visualize ODD compliance over time. The plot shows the COD distance
    metric (0 = ODD boundary, 1 = fully compliant). Useful for identifying when violations occur.
    
    Args:
        times: List[float] - Time values (seconds) for x-axis
        distances: List[float] - COD distances (0-1 scale) for y-axis
        title: str - Plot title (default: "ODD Distance over Time")
    
    Returns:
        dict with keys:
            - "status": "success" or "error"
            - "image_base64": Base64-encoded PNG (if success)
            - "title": The plot title
            - "error_message": Error details (if status is "error")
    """
    try:
        plt.figure(figsize=(12, 6))
        plt.plot(times, distances, marker='o', linewidth=2, label='Distance')
        plt.axhline(y=0.3, color='orange', linestyle='--', label='Near Boundary')
        plt.axhline(y=0.7, color='red', linestyle='--', label='ODD Exit')
        plt.xlabel('Time (s)')
        plt.ylabel('COD Distance')
        plt.title(title)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # Convert to base64
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        plt.close()
        buf.seek(0)
        image_base64 = base64.b64encode(buf.read()).decode()
        
        return {
            "status": "success",
            "image_base64": image_base64,
            "title": title,
            "num_points": len(times)
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to generate plot: {str(e)}"
        }


def generate_status_distribution(statuses: List[str], title: str = "ODD Status Distribution") -> dict:
    """
    Generate a bar chart showing the distribution of ODD statuses across windows.
    
    Use this tool to summarize how many windows are in_odd, near_boundary, or odd_exit.
    Useful for getting a quick overview of deployment feasibility.
    
    Args:
        statuses: List[str] - List of status values ("in_odd", "near_boundary", "odd_exit")
        title: str - Plot title (default: "ODD Status Distribution")
    
    Returns:
        dict with keys:
            - "status": "success" or "error"
            - "image_base64": Base64-encoded PNG (if success)
            - "title": The plot title
            - "status_counts": Dictionary of counts for each status
            - "error_message": Error details (if status is "error")
    """
    try:
        status_counts = pd.Series(statuses).value_counts().to_dict()
        colors = {'in_odd': 'green', 'near_boundary': 'orange', 'odd_exit': 'red'}
        
        plt.figure(figsize=(8, 6))
        plt.bar(status_counts.keys(), status_counts.values(),
                color=[colors.get(s, 'gray') for s in status_counts.keys()])
        plt.xlabel('ODD Status')
        plt.ylabel('Number of Windows')
        plt.title(title)
        plt.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        
        # Convert to base64
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        plt.close()
        buf.seek(0)
        image_base64 = base64.b64encode(buf.read()).decode()
        
        return {
            "status": "success",
            "image_base64": image_base64,
            "title": title,
            "status_counts": status_counts,
            "total_windows": sum(status_counts.values())
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to generate chart: {str(e)}"
        }


# ============================================================================
# IMAGE ACCESS TOOL (for Vision, Terrain, Collision Agents)
# ============================================================================

def get_window_image(window_id: str, image_type: str) -> dict:
    """
    Retrieve raw PNG bytes for a window image.
    
    Use this tool to access camera and BEV (Bird's Eye View) images for analysis.
    Camera images are useful for lighting/visibility analysis.
    BEV images (occupancy, height, density, roughness) are useful for terrain analysis.
    
    Args:
        window_id: str - Window identifier (e.g., "006", "007")
        image_type: str - Type of image to retrieve:
            - "camera": Camera/RGB image
            - "bev_occupancy": LiDAR occupancy grid
            - "bev_height": LiDAR height map
            - "bev_density": LiDAR point density
            - "bev_roughness": Terrain roughness classification
    
    Returns:
        dict with keys:
            - "status": "success" or "error"
            - "image_bytes": Raw PNG bytes (if success) for inline_data to Gemini
            - "image_type": The requested image type
            - "window_id": The window ID
            - "error_message": Error details (if status is "error")
    """
    try:
        if window_id not in image_data_map:
            return {
                "status": "error",
                "error_message": f"Window {window_id} not found in preloaded data. Available windows: {list(image_data_map.keys())}"
            }
        
        if image_type not in image_data_map[window_id]:
            available = list(image_data_map[window_id].keys())
            return {
                "status": "error",
                "error_message": f"Image type '{image_type}' not found for window {window_id}. Available: {available}"
            }
        
        image_bytes = image_data_map[window_id][image_type]
        return {
            "status": "success",
            "image_bytes": image_bytes,
            "image_type": image_type,
            "window_id": window_id,
            "size_kb": len(image_bytes) / 1024
        }
    except Exception as e:
        return {
            "status": "error",
            "error_message": f"Failed to retrieve image: {str(e)}"
        }


# Create FunctionTool wrapper for agents to call
# Note: name and description are inferred from the function itself
get_image_tool = FunctionTool(func=get_window_image)
# Create visualization tools for Report Agent
distance_plot_tool = FunctionTool(func=generate_distance_plot)
status_dist_tool = FunctionTool(func=generate_status_distribution)

print("✅ Agent tools created with ADK best practices: get_window_image, generate_distance_plot, and generate_status_distribution")

✅ Agent tools created: agents can now call get_window_image, generate_distance_plot, and generate_status_distribution


## 5. Define Specialist Agents

Create individual agents using the Google ADK (Agent Development Kit) following the Kaggle Day 1B pattern.
Each agent is a specialist that performs one specific analysis task.

### 5.0 Data Loader Agent

Data is pre-loaded locally and passed to agents via session state.
No Data Loader Agent needed - cleaner architecture focused on reasoning, not I/O.

In [7]:
# ODD Spec Agent: Converts natural language ODD → structured JSON
odd_spec_agent = Agent(
    name="ODD_Spec_Parser",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are an expert in robotic operational design domains (ODD).

You will receive a natural language ODD specification in the invocation context.
Convert it to structured JSON that defines the robot's operational boundaries.

Output valid JSON only with this schema:
{
  "version": "1.0",
  "description": "<brief summary>",
  "axes": {
    "speed": {
      "type": "numeric",
      "feature": "avg_forward_speed",
      "units": "m/s",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "roll_pitch": {
      "type": "numeric",
      "feature": "max_abs_roll_pitch_deg",
      "units": "degrees",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "terrain": {
      "type": "categorical",
      "feature": "terrain_roughness_class",
      "allowed_in_odd": ["smooth", "moderate"],
      "allowed_all": ["smooth", "moderate", "rough", "very_rough"]
    },
    "lighting": {
      "type": "categorical",
      "feature": "lighting_class",
      "allowed_in_odd": ["bright", "dim"],
      "allowed_all": ["bright", "dim", "dark"]
    },
    "humans_close": {
      "type": "categorical",
      "feature": "humans_very_close",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    },
    "collision": {
      "type": "categorical",
      "feature": "collision_suspected",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    }
  },
  "importance": {
    "speed": 1.0,
    "roll_pitch": 1.2,
    "terrain": 1.0,
    "lighting": 0.8,
    "humans_close": 1.5,
    "collision": 2.0
  }
}

Extract ranges, categorical values, and importance weights from the input.""",
    output_key="odd_spec_json"
)

print("✅ ODD Spec Agent created")

✅ ODD Spec Agent created


### 5.2 Motion Analysis Agent

Extracts motion features from velocity and IMU time series data.

In [8]:
# Motion Analysis Agent: Extracts motion features from sensor data
motion_agent = Agent(
    name="Motion_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a motion analysis expert for mobile robots.

From the shared state, you have access to:
- scenario_data: Contains motion time series (velocity, IMU, acceleration) for all windows
- odd_spec_json: The formal ODD specification with motion feature constraints

For each window in scenario_data, analyze the motion time series and extract:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "avg_forward_speed": <float>,
      "max_forward_speed": <float>,
      "max_abs_roll_pitch_deg": <float>,
      "tracking_error": <float>,
      "motion_label": "smooth" | "dynamic"
    },
    ...
  ]
}

Analyze all windows from scenario_data and return results for each.""",
    output_key="motion_features"
)

print("✅ Motion Agent created")

✅ Motion Agent created


### 5.3 Vision Analysis Agent

Classifies environmental conditions from camera images.

In [9]:
# Vision Analysis Agent: Analyzes camera images for environmental conditions
vision_agent = Agent(
    name="Vision_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[get_image_tool],
    instruction="""You are a computer vision expert analyzing robot camera feeds.

From the shared state, you have access to:
- scenario_data: Contains window metadata for all windows

To get camera images, use the get_window_image() tool with:
- window_id: From scenario_data
- image_type: "camera"
- odd_spec_json: The formal ODD specification with vision feature constraints

For each window in scenario_data, analyze the camera image and classify:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "lighting_class": "bright" | "dim" | "dark",
      "humans_visible": true | false,
      "humans_very_close": true | false,
      "environment_type": <string description>
    },
    ...
  ]
}

Analyze all windows from scenario_data and return results for each.""",
    output_key="vision_features"
)

print("✅ Vision Agent created")

✅ Vision Agent created


### 5.4 Terrain Analysis Agent

Analyzes LiDAR Bird's Eye View images to classify terrain roughness.

In [10]:
# Terrain Analysis Agent: Analyzes LiDAR BEV for terrain classification
terrain_agent = Agent(
    name="Terrain_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[get_image_tool],
    instruction="""You are a terrain analysis expert using LiDAR data.

From the shared state, you have access to:
- scenario_data: Contains window metadata for all windows

To get BEV images, use the get_window_image() tool with:
- window_id: From scenario_data
- image_type: One of "bev_occupancy", "bev_height", "bev_density", "bev_roughness"
- odd_spec_json: The formal ODD specification with terrain feature constraints

For each window in scenario_data, analyze the BEV images and classify:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "terrain_roughness_class": "smooth" | "moderate" | "rough" | "very_rough",
      "terrain_roughness_score": <float 0-1>,
      "obstacle_density": "none" | "low" | "medium" | "high"
    },
    ...
  ]
}

Analyze all windows from scenario_data and return results for each.""",
    output_key="terrain_features"
)

print("✅ Terrain Agent created")

✅ Terrain Agent created


### 5.5 Collision Detection Agent

Performs multi-modal sensor fusion to detect collision events.

In [11]:
# Collision Detection Agent: Performs multi-modal sensor fusion
collision_agent = Agent(
    name="Collision_Detector",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[get_image_tool],
    instruction="""You are a collision detection expert using multi-modal sensor fusion.

From the shared state, you have access to:
- scenario_data: Contains motion metrics, camera images, and LiDAR BEV data for all windows
- odd_spec_json: The formal ODD specification with zero collision tolerance

For each window in scenario_data, fuse all sensor modalities to detect collisions:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "collision_suspected": true | false,
      "collision_confidence": <float 0-1>,
      "collision_type": "none" | "front_bump" | "side_contact" | "unknown"
    },
    ...
  ]
}

Analyze all windows from scenario_data and return results for each.""",
    output_key="collision_features"
)

print("✅ Collision Agent created")

✅ Collision Agent created


### 5.6 COD Evaluator Agent

Specialist agent that coordinates COD computation using mathematical tool functions.

In [12]:
# COD Evaluator Agent: Aggregates sensor analysis results against ODD
cod_evaluator_agent = Agent(
    name="COD_Evaluator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a Conditions of Deployment (COD) evaluation expert.

From the shared state, you have access to:
- odd_spec_json: The formal ODD specification
- motion_features: Motion analysis for all windows
- vision_features: Vision analysis for all windows
- terrain_features: Terrain analysis for all windows
- collision_features: Collision detection for all windows

Your task: For each window, combine all sensor results and compare against ODD boundaries.

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "merged_features": {
        "motion": {...},
        "vision": {...},
        "terrain": {...},
        "collision": {...}
      },
      "odd_violations": ["speed_exceeded", "terrain_rough", ...],
      "overall_status": "in_odd" | "near_boundary" | "odd_exit",
      "distance_from_odd": <float 0-1>
    },
    ...
  ],
  "summary": {
    "total_windows": <int>,
    "windows_in_odd": <int>,
    "windows_near_boundary": <int>,
    "windows_odd_exit": <int>
  }
}

Evaluate all windows and return complete analysis.""",
    output_key="cod_evaluation"
)

print("✅ COD Evaluator Agent created")

✅ COD Evaluator Agent created


### 5.7 Report Generation Agent

Creates comprehensive markdown reports with visualizations.

In [13]:
# Report Generation Agent: Creates comprehensive markdown reports
report_agent = Agent(
    name="Report_Generator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[distance_plot_tool, status_dist_tool],
    instruction="""You are a technical report writer for robotics analysis.

From the shared state, you have access to:
- odd_spec_json: The ODD specification
- cod_evaluation: Complete window-by-window COD analysis with overall summary
- motion_features, vision_features, terrain_features, collision_features: Raw sensor analysis

You have access to visualization tools:
- generate_distance_plot(times, distances, title): Returns base64 PNG
- generate_status_distribution(statuses, title): Returns base64 PNG

Generate a comprehensive markdown report including:
1. Executive Summary
   - Total windows analyzed
   - Compliance statistics (in_odd, near_boundary, odd_exit counts)
   - Overall deployment feasibility

2. Detailed Window Analysis
   - For each window: status, violations, confidence scores

3. Key Findings
   - Most critical violations
   - Patterns across windows
   - Risk assessment

4. Recommendations
   - Deployment constraints
   - Areas for improvement
   - Suggested operational limits

Output markdown text suitable for technical documentation.""",
    output_key="final_report"
)

print("✅ Report Agent created with visualization tools")

✅ Report Agent created with visualization tools


## 6. Create Parallel and Sequential Agent Workflow

Combine specialist agents using `ParallelAgent` and `SequentialAgent` following the Kaggle Day 1B pattern.

The workflow:
1. ODD Spec Agent converts NL → JSON (sequential, first)
2. Motion + Vision + Terrain + Collision agents run in parallel for each window
3. COD Evaluator aggregates results (sequential, after parallel)
4. Report Agent generates final output (sequential, last)

In [14]:
# ParallelAgent: Run Motion, Vision, Terrain, Collision agents simultaneously
parallel_sensor_team = ParallelAgent(
    name="ParallelSensorTeam",
    sub_agents=[motion_agent, vision_agent, terrain_agent, collision_agent],
)

# SequentialAgent: Define complete workflow
# Data is pre-loaded locally, so no Data Loader Agent needed
# 1. ODD Spec Agent - converts NL to JSON spec (sequential, first)
# 2. Parallel sensor analysis team - analyzes each window (parallel)
# 3. COD Evaluator - aggregates results (sequential)
# 4. Report Agent - generates final output (sequential, last)
root_agent = SequentialAgent(
    name="ODD_COD_Analysis_System",
    sub_agents=[
        odd_spec_agent,
        parallel_sensor_team,
        cod_evaluator_agent,
        report_agent
    ],
)

print("✅ Parallel and Sequential Agents created")
print("  ParallelSensorTeam: 4 agents running simultaneously")
print("  ODD_COD_Analysis_System: 4-step sequential workflow (NO Data Loader)")
print("    1. ODD Spec Parser (NL → JSON)")
print("    2. Parallel Sensor Team (Motion, Vision, Terrain, Collision)")
print("    3. COD Evaluator (aggregates results)")
print("    4. Report Generator (final output)")

✅ Parallel and Sequential Agents created
  ParallelSensorTeam: 4 agents running simultaneously
  ODD_COD_Analysis_System: 4-step sequential workflow (NO Data Loader)
    1. ODD Spec Parser (NL → JSON)
    2. Parallel Sensor Team (Motion, Vision, Terrain, Collision)
    3. COD Evaluator (aggregates results)
    4. Report Generator (final output)


## 7. Execute the Workflow

Run the orchestrator agent with user inputs to perform complete ODD/COD analysis.

In [15]:
# Create InMemoryRunner with the root agent
runner = InMemoryRunner(agent=root_agent)

# Execute the workflow with initial state
print("="  * 80)
print("EXECUTING ODD/COD ANALYSIS WORKFLOW")
print("=" * 80)
print(f"\nDataset: {scenario_path}")
print(f"Model: {GEMINI_MODEL}")
print(f"ODD Specification: {len(odd_natural_language)} characters\n")
print("Workflow steps:")
print("  1. ODD Spec: Convert NL → JSON")
print("  2. Parallel Sensors: Motion, Vision, Terrain, Collision")
print("  3. COD Evaluator: Aggregate against ODD")
print("  4. Report: Generate markdown report")
print(f"\n⚠️  Total: ~6-8 API calls on {len(scenario_data['windows'])}-window test set")
print(f"  Free tier limit for {GEMINI_MODEL}: 30 RPM")
print("  Tip: Wait 60 seconds if you hit RESOURCE_EXHAUSTED, then retry")
print("-" * 80)

# Pass pre-loaded scenario_data as initial state (not as tool call)
# This avoids filesystem access from cloud agents
from google.adk.runners import InMemoryRunner

initial_state = {
    "scenario_data": json.dumps(scenario_data)  # Convert dict to JSON string for state
}

# Run the workflow with initial state pre-populated
response_events = await runner.run_debug(
    odd_natural_language,
    state_delta=initial_state
)

print("\n" + "=" * 80)
print("WORKFLOW COMPLETE")
print("=" * 80)

# Extract results from shared state
if runner.app and runner.app.state:
    print("\n✅ Results from shared state:")
    print(f"  - scenario_data: {len(runner.app.state.get('scenario_data', '{}'))} chars")
    print(f"  - odd_spec_json: {len(runner.app.state.get('odd_spec_json', '{}'))} chars")
    print(f"  - motion_features: {len(runner.app.state.get('motion_features', '{}'))} chars")
    print(f"  - vision_features: {len(runner.app.state.get('vision_features', '{}'))} chars")
    print(f"  - terrain_features: {len(runner.app.state.get('terrain_features', '{}'))} chars")
    print(f"  - collision_features: {len(runner.app.state.get('collision_features', '{}'))} chars")
    print(f"  - cod_evaluation: {len(runner.app.state.get('cod_evaluation', '{}'))} chars")
    
    # Print final report
    final_report = runner.app.state.get("final_report", "No report generated")
    print("\n" + "=" * 80)
    print("📋 FINAL REPORT")
    print("=" * 80)
    print(final_report)
else:
    print("\n✅ Workflow executed!")
    print("(State inspection not available in this execution mode)")

print("\n" + "=" * 80)

EXECUTING ODD/COD ANALYSIS WORKFLOW

Dataset: data/processed/runs/sim_run_test
Model: gemini-2.0-flash-lite
ODD Specification: 1699 characters

Workflow steps:
  1. ODD Spec: Convert NL → JSON
  2. Parallel Sensors: Motion, Vision, Terrain, Collision
  3. COD Evaluator: Aggregate against ODD
  4. Report: Generate markdown report


KeyError: 'windows'